In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import warnings
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.ndimage import (
    binary_fill_holes, distance_transform_edt, gaussian_filter,
    label, generate_binary_structure, binary_dilation
)
from skimage.morphology import skeletonize

warnings.filterwarnings('ignore')

# =============================================================================
# MULTI-GPU DETECTION
# =============================================================================
N_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

@dataclass
class Config:
    # Paths
    TEST_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-surface-detection/test_images")
    CHECKPOINT_PATH: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/6/checkpoints_v11/fold_0/best_model.pth")
    OUTPUT_DIR: Path = Path("/kaggle/working")
    
    # Model (must match training)
    TRAIN_PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])
    
    # Inference - MATCH TRAINING!
    INFER_PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)  # Same as training!
    OVERLAP: float = 0.7
    TTA_LEVEL: str = "flip"
    USE_FLOAT16: bool = True
    
    # Multi-GPU settings
    USE_MULTI_GPU: bool = True
    BATCH_SIZE: int = 1 * N_GPUS  # 1 patch per GPU (192³ is large)
    
    # Post-processing - MATCH BASELINE!
    THRESHOLD: float = 0.70  # Baseline uses 0.70, not 0.50!
    
    DEVICE: str = "cuda"

cfg = Config()
print(f"\nConfiguration:")
print(f"  Inference patch: {cfg.INFER_PATCH_SIZE} (same as training)")
print(f"  Batch size: {cfg.BATCH_SIZE} (across {N_GPUS} GPUs)")
print(f"  Threshold: {cfg.THRESHOLD}")
print(f"  TTA: {cfg.TTA_LEVEL}")

# Clear GPU memory on all devices
for i in range(N_GPUS):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

Available GPUs: 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)

Configuration:
  Inference patch: (192, 192, 192) (same as training)
  Batch size: 2 (across 2 GPUs)
  Threshold: 0.7
  TTA: flip


110

In [2]:
# =============================================================================
# MODEL ARCHITECTURE (same as training)
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1

class HybridConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True))
    def forward(self, x): return self.conv(x)

class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList([ConvBlock(self.width, self.width, use_hybrid) for _ in range(scales-1)])
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i+1] + outputs[-1]) if i > 0 else conv(splits[i+1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))

class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(self.conv_spatial(torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)))
        return x_ca * sa

class SurfaceRefinementBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True))
    def forward(self, x):
        return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))

class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, 2, use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i]*2, features[i]))
            else:
                self.dec_convs.append(nn.Sequential(
                    ConvBlock(features[i]*2, features[i], use_hybrid_conv),
                    MultiScaleResBlock(features[i], 2, use_hybrid_conv)))
        self.final = nn.Conv3d(features[0], out_ch, 1)
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        enc_features = enc_features[::-1]
        x = enc_features[0]
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.final(x)

print("Model loaded")

Model loaded


In [ ]:
# =============================================================================
# MULTI-GPU INFERENCE - MATCHING V11 TRAINING EXACTLY
# =============================================================================

def robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5):
    """
    From V11 training notebook - percentile clipping + Z-score.
    """
    p_low = np.percentile(img, lower_percentile)
    p_high = np.percentile(img, upper_percentile)
    img_clipped = np.clip(img, p_low, p_high)
    mean = img_clipped.mean()
    std = img_clipped.std()
    img_norm = (img_clipped - mean) / (std + 1e-8)
    return img_norm.astype(np.float32)


def create_gaussian_weight(patch_size, sigma=0.125):
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d/2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h/2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w/2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)


def get_patch_positions(volume_shape, patch_size, overlap=0.5):
    """Generate all patch positions for the volume."""
    D, H, W = volume_shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd*(1-overlap)), int(ph*(1-overlap)), int(pw*(1-overlap))
    
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if len(z_pos) == 0 or z_pos[-1] + pd < D: z_pos.append(max(0, D - pd))
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if len(y_pos) == 0 or y_pos[-1] + ph < H: y_pos.append(max(0, H - ph))
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if len(x_pos) == 0 or x_pos[-1] + pw < W: x_pos.append(max(0, W - pw))
    
    positions = []
    for z in z_pos:
        for y in y_pos:
            for x in x_pos:
                positions.append((z, y, x))
    return positions


@torch.no_grad()
def sliding_window_inference_multigpu(model, volume, patch_size, overlap=0.5, batch_size=2):
    """
    Multi-GPU sliding window inference.
    """
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    
    # Pad if needed
    pad_d, pad_h, pad_w = max(0, pd-D), max(0, ph-H), max(0, pw-W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume = np.pad(volume, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
        D, H, W = volume.shape
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)
    
    # Get all patch positions
    positions = get_patch_positions((D, H, W), patch_size, overlap)
    
    # Normalize volume EXACTLY as in training
    vol_norm = robust_zscore_normalize(volume, lower_percentile=0.5, upper_percentile=99.5)
    
    print(f"  Total patches: {len(positions)}, Batch size: {batch_size}")
    
    # Process in batches
    for batch_start in tqdm(range(0, len(positions), batch_size), desc="Inference"):
        batch_end = min(batch_start + batch_size, len(positions))
        batch_positions = positions[batch_start:batch_end]
        
        # Extract patches for this batch
        patches = []
        for (z, y, x) in batch_positions:
            patch = vol_norm[z:z+pd, y:y+ph, x:x+pw]
            patches.append(patch)
        
        # Stack into batch tensor [B, 1, D, H, W]
        batch_tensor = torch.from_numpy(np.stack(patches)[:, None]).cuda().half()
        
        # Forward pass
        with torch.cuda.amp.autocast(dtype=torch.float16):
            batch_pred = torch.sigmoid(model(batch_tensor))
        
        # Convert back to numpy
        batch_pred = batch_pred.squeeze(1).float().cpu().numpy()
        
        # Accumulate predictions
        for i, (z, y, x) in enumerate(batch_positions):
            pred_sum[z:z+pd, y:y+ph, x:x+pw] += batch_pred[i] * gauss
            weight_sum[z:z+pd, y:y+ph, x:x+pw] += gauss
        
        # Cleanup
        del batch_tensor, batch_pred, patches
        
        # Periodic GPU cleanup
        if (batch_start // batch_size) % 10 == 0:
            torch.cuda.empty_cache()
    
    pred = pred_sum / np.maximum(weight_sum, 1e-8)
    
    # Remove padding
    if pad_d > 0: pred = pred[:-pad_d]
    if pad_h > 0: pred = pred[:, :-pad_h]
    if pad_w > 0: pred = pred[:, :, :-pad_w]
    
    return pred


@torch.no_grad()
def inference_with_tta_multigpu(model, volume, patch_size, overlap=0.5, batch_size=2, tta='flip'):
    """TTA with multi-GPU support."""
    # Original
    print("  TTA: Original")
    pred = sliding_window_inference_multigpu(model, volume, patch_size, overlap, batch_size)
    
    if tta == 'none':
        return pred
    
    preds = [pred]
    del pred
    gc.collect()
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
    
    if tta in ['flip', 'full']:
        for axis in [0, 1, 2]:
            print(f"  TTA: Flip axis {axis}")
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference_multigpu(model, vol_flip, patch_size, overlap, batch_size)
            preds.append(np.flip(pred_flip, axis).copy())
            
            del vol_flip, pred_flip
            gc.collect()
            for i in range(N_GPUS):
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
    
    print(f"  TTA: Averaging {len(preds)} predictions")
    result = np.mean(preds, axis=0)
    del preds
    gc.collect()
    
    return result

print("Inference functions ready (MATCHING V11 TRAINING)")
print(f"  Normalization: Percentile clipping (0.5-99.5%) + Z-score")
print(f"  Patch size: {cfg.INFER_PATCH_SIZE}")

Inference functions ready (MATCHING V11 TRAINING)
  Normalization: Percentile clipping (0.5-99.5%) + Z-score
  Patch size: (192, 192, 192)


In [4]:
# =============================================================================
# POST-PROCESSING (OPTIMIZED FOR BEST SCORE)
# =============================================================================
# Changes from previous version:
# - Removed Frangi filter (was hurting score - mean dropped from 0.118 to 0.089)
# - Fixed threshold 0.5 instead of adaptive (0.30 was too aggressive)
# - Kept 2D slicewise operations (gentle, topology-safe)

from scipy.ndimage import (
    binary_closing, binary_opening, binary_fill_holes,
    label, generate_binary_structure, binary_dilation, binary_erosion
)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def count_components(mask):
    """Count connected components using 26-connectivity."""
    struct = generate_binary_structure(3, 3)
    _, n = label(mask, structure=struct)
    return n


def remove_small_components(mask, min_size=50):
    """Remove components smaller than min_size voxels."""
    struct = generate_binary_structure(3, 3)
    labeled, n = label(mask, structure=struct)
    if n == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    small = sizes < min_size
    small[0] = False
    result = mask.copy()
    result[small[labeled]] = 0
    return result


def topology_safe_operation(mask, operation_func, name="op"):
    """Apply operation only if it doesn't REDUCE component count."""
    n_before = count_components(mask)
    result = operation_func(mask)
    n_after = count_components(result)
    
    if n_after < n_before:
        print(f"    [REVERT] {name}: would merge {n_before}->{n_after} components")
        return mask
    return result


# =============================================================================
# 2D SLICE-WISE OPERATIONS (gentle, preserves thin structures)
# =============================================================================

def slicewise_hole_fill(mask):
    """Fill holes slice-by-slice in all 3 axes."""
    filled = mask.copy()
    for i in range(mask.shape[0]):
        filled[i] = binary_fill_holes(filled[i])
    for i in range(mask.shape[1]):
        filled[:, i, :] = binary_fill_holes(filled[:, i, :])
    for i in range(mask.shape[2]):
        filled[:, :, i] = binary_fill_holes(filled[:, :, i])
    return filled


def slicewise_morphology(mask, operation='close', iterations=1):
    """Apply morphological operations slice-by-slice."""
    result = mask.copy()
    struct_2d = generate_binary_structure(2, 1)  # 4-connectivity (gentler)
    
    for axis in range(3):
        for i in range(mask.shape[axis]):
            if axis == 0:
                slc = result[i]
            elif axis == 1:
                slc = result[:, i, :]
            else:
                slc = result[:, :, i]
            
            if operation == 'close':
                slc_new = binary_closing(slc, structure=struct_2d, iterations=iterations)
            elif operation == 'open':
                slc_new = binary_opening(slc, structure=struct_2d, iterations=iterations)
            else:
                slc_new = slc
            
            if axis == 0:
                result[i] = slc_new
            elif axis == 1:
                result[:, i, :] = slc_new
            else:
                result[:, :, i] = slc_new
    
    return result


# =============================================================================
# MAIN POST-PROCESSING PIPELINE (OPTIMIZED)
# =============================================================================

def postprocess_v11(pred_prob, 
                    threshold=0.5,  # Fixed threshold (not adaptive)
                    min_component_size=50,
                    use_morphology=True,
                    use_hole_fill=True,
                    verbose=True):
    """
    Optimized post-processing pipeline.
    
    Changes for better score:
    - NO Frangi filter (was reducing mean probability)
    - Fixed threshold 0.5 (adaptive 0.30 was too aggressive)
    - 2D slice-wise morphology (gentle)
    - Topology-safe operations
    """
    if verbose:
        print("Post-processing (OPTIMIZED):")
        print(f"  Input: min={pred_prob.min():.3f}, max={pred_prob.max():.3f}, mean={pred_prob.mean():.3f}")
    
    # Step 1: Fixed threshold (0.5 is standard)
    mask = (pred_prob > threshold).astype(np.uint8)
    fg_pct = 100 * mask.mean()
    if verbose:
        print(f"  1. Threshold ({threshold}): {mask.sum():,} voxels, FG={fg_pct:.2f}%")
    
    if mask.sum() == 0:
        if verbose:
            print("  WARNING: Empty mask!")
        return mask
    
    # Step 2: Remove small components
    n_before = count_components(mask)
    mask = remove_small_components(mask, min_component_size)
    n_after = count_components(mask)
    if verbose:
        print(f"  2. Remove small (<{min_component_size}): {n_before}->{n_after} components")
    
    # Step 3: 2D slice-wise closing (topology-safe)
    if use_morphology:
        mask = topology_safe_operation(
            mask, 
            lambda m: slicewise_morphology(m, 'close', iterations=1),
            "slicewise_close"
        )
        if verbose:
            print(f"  3. Slicewise closing: FG={100*mask.mean():.2f}%")
    
    # Step 4: 2D slice-wise hole filling (topology-safe)
    if use_hole_fill:
        mask = topology_safe_operation(mask, slicewise_hole_fill, "slicewise_hole_fill")
        if verbose:
            print(f"  4. Slicewise hole fill: FG={100*mask.mean():.2f}%")
    
    # Step 5: 2D slice-wise opening (topology-safe) - cleanup
    if use_morphology:
        mask = topology_safe_operation(
            mask,
            lambda m: slicewise_morphology(m, 'open', iterations=1),
            "slicewise_open"
        )
        if verbose:
            print(f"  5. Slicewise opening: FG={100*mask.mean():.2f}%")
    
    # Step 6: Final cleanup
    mask = remove_small_components(mask, min_component_size)
    n_final = count_components(mask)
    
    if verbose:
        print(f"  Final: {n_final} components, {mask.sum():,} voxels, FG={100*mask.mean():.2f}%")
    
    return mask


print("Post-processing ready (OPTIMIZED)")
print("Changes:")
print("  - NO Frangi filter (was hurting score)")
print("  - Fixed threshold 0.5 (not adaptive)")
print("  - 2D slicewise morphology (gentle)")
print("  - Topology-safe operations")

Post-processing ready (OPTIMIZED)
Changes:
  - NO Frangi filter (was hurting score)
  - Fixed threshold 0.5 (not adaptive)
  - 2D slicewise morphology (gentle)
  - Topology-safe operations


In [5]:
# =============================================================================
# SUBMISSION FORMAT: TIFF MASKS
# =============================================================================

import zipfile

def save_mask_tiff(mask, output_path):
    """Save mask as TIFF - NO compression to match expected file size."""
    mask_uint8 = mask.astype(np.uint8)
    # No compression - raw TIFF
    tifffile.imwrite(str(output_path), mask_uint8, compression=None)
    print(f"  Saved: {output_path} ({mask_uint8.shape}, dtype={mask_uint8.dtype})")

def create_submission_zip(mask_dir, output_zip):
    """Create submission.zip containing all mask TIFFs."""
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for tif_path in sorted(Path(mask_dir).glob("*.tif")):
            zf.write(tif_path, tif_path.name)
            print(f"  Added to zip: {tif_path.name}")
    print(f"Submission zip: {output_zip}")

print("TIFF submission functions loaded (no compression)")

TIFF submission functions loaded (no compression)


In [6]:
# =============================================================================
# LOAD MODEL WITH MULTI-GPU (DataParallel)
# =============================================================================

# Clear all GPU memory
for i in range(N_GPUS):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
gc.collect()

print(f"Loading: {cfg.CHECKPOINT_PATH}")

# Create model on primary GPU first
model = TopoPreservingUNet3D(features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS).cuda()

# Load checkpoint
ckpt = torch.load(cfg.CHECKPOINT_PATH, map_location='cuda:0', weights_only=False)
state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
model.load_state_dict(state, strict=False)

print(f"  Epoch: {ckpt.get('epoch', '?')}, Best Dice: {ckpt.get('best_dice', '?'):.4f}")

# Wrap with DataParallel for multi-GPU
if N_GPUS > 1 and cfg.USE_MULTI_GPU:
    print(f"\n>>> Enabling DataParallel across {N_GPUS} GPUs")
    model = nn.DataParallel(model, device_ids=list(range(N_GPUS)))
    print(f"  Device IDs: {list(range(N_GPUS))}")
else:
    print("\n>>> Single GPU mode")

# Convert to half precision
model.half()
model.eval()

# Memory report
print("\nGPU Memory Usage:")
for i in range(N_GPUS):
    mem = torch.cuda.memory_allocated(i) / 1e9
    print(f"  GPU {i}: {mem:.2f} GB")

print("\nModel ready for inference!")

Loading: /kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/6/checkpoints_v11/fold_0/best_model.pth
  Epoch: 394, Best Dice: 0.5785

>>> Enabling DataParallel across 2 GPUs
  Device IDs: [0, 1]

GPU Memory Usage:
  GPU 0: 0.14 GB
  GPU 1: 0.00 GB

Model ready for inference!


In [7]:
# =============================================================================
# RUN INFERENCE (MULTI-GPU) - OPTIMIZED POST-PROCESSING
# =============================================================================

test_files = sorted(cfg.TEST_ROOT.glob("*.tif")) + sorted(cfg.TEST_ROOT.glob("*.tiff"))
print(f"Found {len(test_files)} test volumes")

# Create output directory for masks
mask_dir = cfg.OUTPUT_DIR / "masks"
mask_dir.mkdir(exist_ok=True)

for vol_path in test_files:
    vol_id = vol_path.stem
    print(f"\n{'='*70}")
    print(f"Processing: {vol_id}")
    print(f"{'='*70}")
    
    # Clear memory on all GPUs before each volume
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
    gc.collect()
    
    # Load volume
    volume = tifffile.imread(str(vol_path)).astype(np.float32)
    original_shape = volume.shape
    print(f"Shape: {original_shape}")
    
    # Inference with multi-GPU function
    print(f"Running inference (patch={cfg.INFER_PATCH_SIZE}, TTA={cfg.TTA_LEVEL}, batch={cfg.BATCH_SIZE})...")
    pred_prob = inference_with_tta_multigpu(
        model, volume, cfg.INFER_PATCH_SIZE, cfg.OVERLAP,
        cfg.BATCH_SIZE, cfg.TTA_LEVEL
    )
    
    del volume
    gc.collect()
    
    # OPTIMIZED Post-processing:
    # - NO Frangi (was hurting score)
    # - Fixed threshold 0.5 (not adaptive 0.30)
    # - 2D slicewise morphology
    pred_mask = postprocess_v11(
        pred_prob,
        threshold=0.5,  # Fixed threshold
        min_component_size=50,
        use_morphology=True,
        use_hole_fill=True,
        verbose=True
    )
    
    # Verify shape matches original
    assert pred_mask.shape == original_shape, f"Shape mismatch: {pred_mask.shape} vs {original_shape}"
    
    # Save as TIFF (no compression)
    mask_path = mask_dir / f"{vol_id}.tif"
    save_mask_tiff(pred_mask, mask_path)
    
    # Check file size
    actual_size = mask_path.stat().st_size / 1e6
    print(f"  Saved: {mask_path.name} ({actual_size:.2f} MB)")
    
    # Cleanup
    del pred_prob, pred_mask
    gc.collect()
    for i in range(N_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()

print(f"\n{'='*70}")
print("INFERENCE COMPLETE")
print(f"{'='*70}")

Found 1 test volumes

Processing: 1407735
Shape: (320, 320, 320)
Running inference (patch=(192, 192, 192), TTA=flip, batch=2)...
  TTA: Original
  Total patches: 64, Batch size: 2


Inference:   0%|          | 0/32 [00:00<?, ?it/s]

  TTA: Flip axis 0
  Total patches: 64, Batch size: 2


Inference:   0%|          | 0/32 [00:00<?, ?it/s]

  TTA: Flip axis 1
  Total patches: 64, Batch size: 2


Inference:   0%|          | 0/32 [00:00<?, ?it/s]

  TTA: Flip axis 2
  Total patches: 64, Batch size: 2


Inference:   0%|          | 0/32 [00:00<?, ?it/s]

  TTA: Averaging 4 predictions
Post-processing (OPTIMIZED):
  Input: min=0.000, max=0.998, mean=0.089
  1. Threshold (0.5): 2,819,068 voxels, FG=8.60%
  2. Remove small (<50): 135->12 components
    [REVERT] slicewise_close: would merge 12->11 components
  3. Slicewise closing: FG=8.60%
  4. Slicewise hole fill: FG=8.61%
  5. Slicewise opening: FG=8.14%
  Final: 11 components, 2,668,261 voxels, FG=8.14%
  Saved: /kaggle/working/masks/1407735.tif ((320, 320, 320), dtype=uint8)
  Saved: 1407735.tif (32.82 MB)

INFERENCE COMPLETE


In [8]:
# =============================================================================
# CREATE SUBMISSION ZIP
# =============================================================================

submission_zip = cfg.OUTPUT_DIR / "submission.zip"
create_submission_zip(mask_dir, submission_zip)

# Verify submission
print(f"\nSubmission contents:")
with zipfile.ZipFile(submission_zip, 'r') as zf:
    for info in zf.infolist():
        print(f"  {info.filename}: {info.file_size / 1e6:.2f} MB")

print(f"\n>>> Ready to submit: {submission_zip}")

  Added to zip: 1407735.tif
Submission zip: /kaggle/working/submission.zip

Submission contents:
  1407735.tif: 32.82 MB

>>> Ready to submit: /kaggle/working/submission.zip
